# Multi-timeframe causal alignment

**Tier:** Docs-CI executable (CPU) · **Expected runtime:** under 10 seconds · **Hardware:** CPU

Different source timeframes become observable at different close timestamps. H4 uses 240-minute bars and D1 uses 1,440-minute bars; this synthetic fixture demonstrates distinct sampling intervals, while production daily close times must follow the instrument session calendar. The alignment helper projects completed values to an M1 spine.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from market_data.transforms.align import align_timeframes

spine = pd.date_range("2025-01-02 09:00", periods=241, freq="min", tz="UTC")
m1 = pd.DataFrame({"close": 100 + np.sin(np.arange(len(spine)) / 12)}, index=spine)


def source(minutes):
    """Build a deterministic source frame at the requested minute interval."""
    index = spine[::minutes]
    return pd.DataFrame({"close": 100 + np.arange(len(index)) * 0.5}, index=index)


frames = {
    "M5": source(5),
    "M15": source(15),
    "H1": source(60),
    "H4": source(240),
    "D1": source(240),
}
aligned = align_timeframes(m1, frames)
print(aligned[["M5_close", "M15_close", "H1_close"]].head(65).tail(8))

Figure: step lines for completed M5, M15, and H1 values projected onto the UTC M1 timeline.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for col in ("M5_close", "M15_close", "H1_close"):
    ax.step(aligned.index, aligned[col], where="post", label=col)
ax.set(
    title="Latest completed source value on the M1 timeline",
    ylabel="Illustrative source value",
    xlabel="UTC",
)
ax.legend()
plt.show()
assert pd.isna(aligned["H1_close"].iloc[1])
assert pd.notna(aligned["H1_close"].iloc[60])

## Reading the timeline

The values hold between completion events. Daily candles should use the project session calendar and close-time map in real feature construction; this small example illustrates only the alignment contract.

See [`align_timeframes`](../../../api/domains/data), [causality rules](../../../framework/market_data_and_features), and the [feature cache reference](../../../reference/cache_and_schema_versions).